# 🎬 YouTube to Google Drive Downloader

**دانلود ویدیو و صدای یوتیوب مستقیماً به Google Drive**

Created by [replica](https://github.com/repliiica) • [GitHub Repo](https://github.com/repliiica/RYTD)

---

### 📖 نحوه استفاده:
1. روی `Runtime > Run all` کلیک کنید (یا `Ctrl+F9`)
2. به Google Drive اجازه دسترسی بدهید
3. در فرم‌های سمت راست، تنظیمات را وارد کنید
4. صبر کنید تا دانلود تمام شود ✨

### ⚠️ سلب مسئولیت
این ابزار فقط برای استفاده شخصی و محتوای دارای مجوز است. لطفاً قوانین کپی‌رایت و شرایط استفاده YouTube را رعایت کنید.

## 1️⃣ نصب و راه‌اندازی

In [ ]:
#@title 🔧 نصب yt-dlp و اتصال به Drive
!pip install -q -U yt-dlp

from google.colab import drive
drive.mount('/content/drive')

import os
output_path = '/content/drive/MyDrive/YoutubeDownloads'
os.makedirs(output_path, exist_ok=True)
print(f'✅ آماده! فایل‌ها در این مسیر ذخیره می‌شوند:\n{output_path}')

## 2️⃣ دانلود تکی

In [ ]:
#@title 🎬 تنظیمات دانلود { display-mode: "form" }

#@markdown ### 🔗 لینک ویدیو یا پلی‌لیست:
video_url = "" #@param {type:"string"}

#@markdown ### 📦 نوع دانلود:
media_type = "Video (MP4)" #@param ["Video (MP4)", "Audio (MP3)", "Audio (M4A)"]

#@markdown ### 🎥 کیفیت ویدیو:
video_quality = "1080p" #@param ["360p", "480p", "720p", "1080p", "1440p", "2160p (4K)", "Best Available"]

#@markdown ### 🎵 کیفیت صدا (kbps):
audio_quality = "192" #@param ["128", "192", "256", "320"]

#@markdown ### 📝 آپشن‌های اضافه:
download_subtitles = False #@param {type:"boolean"}
download_thumbnail = False #@param {type:"boolean"}

#@markdown ### ✂️ برش ویدیو (اختیاری):
start_time = "" #@param {type:"string"}
end_time = "" #@param {type:"string"}
#@markdown _فرمت زمان: `00:01:30` یعنی ۱ دقیقه و ۳۰ ثانیه_

if not video_url.strip():
    print("❌ لطفاً لینک ویدیو را وارد کنید!")
else:
    common_args = f'-o "{output_path}/%(title)s.%(ext)s" --no-mtime --restrict-filenames --embed-metadata'
    if download_subtitles:
        common_args += " --write-subs --write-auto-subs --sub-langs 'en,fa' --convert-subs srt --embed-subs"
    if download_thumbnail:
        common_args += " --write-thumbnail --embed-thumbnail"

    trim_args = ""
    if start_time.strip() and end_time.strip():
        trim_args = f'--download-sections "*{start_time}-{end_time}" --force-keyframes-at-cuts'

    quality_map = {"360p":"360","480p":"480","720p":"720","1080p":"1080","1440p":"1440","2160p (4K)":"2160","Best Available":"best"}

    if media_type == "Video (MP4)":
        q = quality_map[video_quality]
        if q == "best":
            fmt = "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best"
        else:
            fmt = f"bestvideo[height<={q}][ext=mp4]+bestaudio[ext=m4a]/best[height<={q}][ext=mp4]/best[height<={q}]"
        cmd = f'yt-dlp -f "{fmt}" --merge-output-format mp4 {trim_args} {common_args} "{video_url}"'
    elif media_type == "Audio (MP3)":
        cmd = f'yt-dlp -x --audio-format mp3 --audio-quality {audio_quality}K {trim_args} {common_args} "{video_url}"'
    else:
        cmd = f'yt-dlp -x --audio-format m4a --audio-quality {audio_quality}K {trim_args} {common_args} "{video_url}"'

    print("="*50)
    print(f"🔗 {video_url}\n📦 {media_type}")
    print("="*50 + "\n⏳ در حال دانلود...\n")
    !{cmd}
    print("\n✅ دانلود تمام شد!")
    print(f"📁 مسیر: {output_path}")

## 3️⃣ دانلود گروهی (Batch)

In [ ]:
#@title 📦 دانلود چند ویدیو با هم { display-mode: "form" }

#@markdown ### کیفیت:
batch_quality = "1080p" #@param ["360p", "480p", "720p", "1080p", "1440p", "2160p (4K)", "Best Available"]
#@markdown ### نوع:
batch_type = "Video (MP4)" #@param ["Video (MP4)", "Audio (MP3)"]

urls = """
# لینک‌ها را اینجا وارد کنید، هر کدام در یک خط:
# https://www.youtube.com/watch?v=xxx
# https://www.youtube.com/watch?v=yyy
"""

url_list = [u.strip() for u in urls.strip().split('\n') if u.strip() and not u.strip().startswith('#')]
if not url_list:
    print("❌ هیچ لینکی وارد نشده!")
else:
    q_map = {"360p":"360","480p":"480","720p":"720","1080p":"1080","1440p":"1440","2160p (4K)":"2160","Best Available":"best"}
    q = q_map[batch_quality]
    print(f"📋 {len(url_list)} ویدیو در صف\n")
    for i, url in enumerate(url_list, 1):
        print(f"\n{'='*50}\n⏳ {i}/{len(url_list)}: {url}\n{'='*50}")
        if batch_type == "Video (MP4)":
            fmt = "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best" if q=="best" else f"bestvideo[height<={q}][ext=mp4]+bestaudio[ext=m4a]/best[height<={q}]"
            !yt-dlp -f "{fmt}" --merge-output-format mp4 -o "{output_path}/%(title)s.%(ext)s" --restrict-filenames "{url}"
        else:
            !yt-dlp -x --audio-format mp3 --audio-quality 192K -o "{output_path}/%(title)s.%(ext)s" --restrict-filenames "{url}"
    print(f"\n✅ همه {len(url_list)} ویدیو دانلود شدند!")